# Fast Food Workout Impact Prediction

## Problem Statement

Fast food consumption affects individuals differently based on meal nutritional content
and personal biometric factors such as age, weight, gender, and fitness level.

This project builds an end-to-end machine learning pipeline to:
1. Predict the **Workout Shock Tier** (classification) — how severe the workout impact will be
2. Predict the **Net Caloric Surplus** (regression) — exact excess calories consumed
3. Analyze nutritional and consumer patterns through exploratory data analysis

## Dataset Description

The dataset contains 8,000 records of fast-food orders with:
- Meal details (chain, category, item count, nutritional breakdown)
- Consumer biometrics (age, gender, weight, fitness level)
- Workout impact metrics (burn times, caloric surplus, shock tier)

## Machine Learning Pipeline

```text
Data Loading → Cleaning → EDA → Feature Selection → Preprocessing →
Model Training → Evaluation → Prediction

In [ ]:
# Core Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, r2_score, mean_squared_error, mean_absolute_error
)

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("All libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-Learn version: {__import__('sklearn').__version__}")
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Configuration
RAW_DATA_PATH = "data/raw/fastfood.csv"
TARGET_CLASSIFICATION = "workout_shock_tier"
TARGET_REGRESSION = "net_caloric_surplus"
RANDOM_STATE = 42

# Helper function for consistent plot formatting
plot_no = 1

def show_fig():
    plt.tight_layout()
    plt.show()

# Set global plot style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Configuration complete.")

---
## 1. Data Loading and Initial Inspection

We begin by loading the dataset and performing a preliminary inspection to understand
its structure, data types, and potential issues.

In [ ]:
# Load dataset
try:
    df = pd.read_csv(RAW_DATA_PATH)
except FileNotFoundError:
    df = pd.read_csv("fastfood.csv")

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn Names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Display first 5 rows
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Dataset information
print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print(f"\nTotal Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print(f"\nData Types:")
print(df.dtypes)
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

In [ ]:
# Check for missing values
print("=" * 60)
print("MISSING VALUES CHECK")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Count'] > 0] if missing.sum() > 0 else "No missing values found.")

# Check for duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Summary statistics
print("\n" + "=" * 60)
print("NUMERICAL SUMMARY")
print("=" * 60)
df.describe()

---
## 2. Data Cleaning

Before analysis, we perform the following cleaning steps:
- Remove unique identifiers (`order_id`) that do not contribute to prediction
- Remove duplicate rows
- Remove rows with missing target values
- Ensure correct data types for all columns
- Strip whitespace from categorical text fields

In [ ]:
# Data cleaning

# Store original shape for comparison
original_shape = df.shape

# Step 1: Remove unique identifier
df = df.drop(columns=['order_id'], errors='ignore')

# Step 2: Remove duplicates
df = df.drop_duplicates()

# Step 3: Remove rows where classification target is missing
df = df.dropna(subset=[TARGET_CLASSIFICATION])

# Step 4: Ensure numeric columns are proper numeric type
numeric_columns = [
    'item_count', 'total_calories', 'fat_grams', 'saturated_fat_grams',
    'carb_grams', 'sugar_grams', 'protein_grams', 'sodium_mg',
    'consumer_age', 'consumer_weight_kg', 'thermic_effect_calories',
    'net_caloric_surplus', 'walking_burn_time_min', 'cycling_burn_time_min',
    'running_burn_time_min', 'hiit_burn_time_min', 'daily_calorie_allowance_pct'
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Step 5: Clean categorical columns
categorical_columns = [
    'fast_food_chain', 'meal_category', 'consumer_gender',
    'fitness_level', TARGET_CLASSIFICATION
]

for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].astype('string').str.strip()

# Step 6: Fill remaining missing numeric values with median
for col in numeric_columns:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(f"Original shape: {original_shape}")
print(f"Cleaned shape:  {df.shape}")
print(f"Rows removed:   {original_shape[0] - df.shape[0]}")
print(f"\nTarget distribution:")
print(df[TARGET_CLASSIFICATION].value_counts())

---
## 3. Exploratory Data Analysis (EDA)

We now explore the relationships between nutritional content, consumer biometrics,
and workout impact. The analysis is organized into four themes:

1. **Nutritional Analysis** — How macros relate to total calories
2. **Consumer Demographics** — How age, weight, and fitness level affect outcomes
3. **Workout Compensation** — How much exercise is needed to burn off meals
4. **Correlation Analysis** — Statistical relationships between all variables

In [ ]:
# EDA Plot 1: Protein Content vs Total Calories by Fitness Level
fig = plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='protein_grams', y='total_calories',
                hue='fitness_level', palette='viridis', alpha=0.7)
plt.title(f'{plot_no}. Protein Content vs Total Calories by Fitness Level')
plt.xlabel('Protein (grams)')
plt.ylabel('Total Calories')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 2: Dietary Fat vs Total Calories by Meal Category
fig = plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='fat_grams', y='total_calories',
                hue='meal_category', palette='Set2', alpha=0.7)
plt.title(f'{plot_no}. Relationship Between Dietary Fat and Total Calories')
plt.xlabel('Fat (grams)')
plt.ylabel('Total Calories')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 3: Sugar Content vs Total Calorie Load by Fast Food Chain
fig = plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='sugar_grams', y='total_calories',
                hue='fast_food_chain', alpha=0.7)
plt.title(f'{plot_no}. Sugar Content vs Total Calorie Load')
plt.xlabel('Sugar (grams)')
plt.ylabel('Total Calories')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 4: Saturated Fat vs Total Fat with Calorie Load (Bubble Chart)
fig = plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='saturated_fat_grams', y='fat_grams',
                size='total_calories', hue='fast_food_chain',
                sizes=(20, 200), alpha=0.6)
plt.title(f'{plot_no}. Saturated Fat vs Total Fat (Bubble Size = Calorie Load)')
plt.xlabel('Saturated Fat (grams)')
plt.ylabel('Total Fat (grams)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 5: Body Weight vs Net Caloric Surplus by Fitness Level
fig = plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='consumer_weight_kg', y='net_caloric_surplus',
                hue='fitness_level', palette='viridis', alpha=0.7)
plt.title(f'{plot_no}. Body Weight vs Net Caloric Surplus by Fitness Level')
plt.xlabel('Consumer Weight (kg)')
plt.ylabel('Net Caloric Surplus (kcal)')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 6: Total Calories vs Net Caloric Surplus by Fitness Level
fig = plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='total_calories', y='net_caloric_surplus',
                hue='fitness_level', palette='viridis', alpha=0.7)
plt.title(f'{plot_no}. Total Calories vs Net Caloric Surplus by Fitness Level')
plt.xlabel('Total Calories')
plt.ylabel('Net Caloric Surplus (kcal)')
show_fig()
plot_no += 1

---
## 4. Workout Compensation Analysis

This section analyzes the physical exercise required to offset fast-food meals.
We examine how different workout intensities relate to caloric surplus and
how fitness level affects the required burn time.

In [ ]:
# EDA Plot 7: Net Caloric Surplus vs Running Time Required
fig = plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='net_caloric_surplus', y='running_burn_time_min',
                hue='workout_shock_tier', palette='viridis', alpha=0.7)
plt.title(f'{plot_no}. Net Caloric Surplus vs Running Time Required for Compensation')
plt.xlabel('Net Caloric Surplus (kcal)')
plt.ylabel('Running Time (minutes)')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 8: Net Caloric Surplus Across Workout Shock Tiers
fig = plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='workout_shock_tier', y='net_caloric_surplus', palette='Set2')
plt.xticks(rotation=20)
plt.title(f'{plot_no}. Net Caloric Surplus Across Workout Shock Tiers')
plt.xlabel('Workout Shock Tier')
plt.ylabel('Net Caloric Surplus (kcal)')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 9: Average Walking Time Required by Fitness Level
fig = plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='fitness_level', y='walking_burn_time_min',
            estimator='mean', palette='viridis')
plt.title(f'{plot_no}. Average Walking Time Required by Fitness Level')
plt.xlabel('Fitness Level')
plt.ylabel('Walking Time (minutes)')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 10: Average Running Time Required to Offset Intake
fig = plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='fitness_level', y='running_burn_time_min',
            estimator='mean', palette='viridis')
plt.title(f'{plot_no}. Average Running Time Required to Offset Intake')
plt.xlabel('Fitness Level')
plt.ylabel('Running Time (minutes)')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 11: Total Calories Distribution by Consumer Gender
fig = plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='consumer_gender', y='total_calories', palette='Set2')
plt.title(f'{plot_no}. Total Calories Distribution by Consumer Gender')
plt.xlabel('Gender')
plt.ylabel('Total Calories')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 12: Average Calories by Meal Category
fig = plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='meal_category', y='total_calories',
            estimator='mean', palette='Set2')
plt.xticks(rotation=20)
plt.title(f'{plot_no}. Average Calories by Meal Category')
plt.xlabel('Meal Category')
plt.ylabel('Total Calories')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 13: Distribution of Total Calories
fig = plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='total_calories', kde=True, bins=50, color='steelblue')
plt.title(f'{plot_no}. Distribution of Total Calories')
plt.xlabel('Total Calories')
plt.ylabel('Frequency')
show_fig()
plot_no += 1

In [ ]:
# EDA Plot 14: Daily Calorie Allowance Percentage by Workout Shock Tier
fig = plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='workout_shock_tier', y='daily_calorie_allowance_pct', palette='Set2')
plt.xticks(rotation=20)
plt.title(f'{plot_no}. Daily Calorie Allowance % by Workout Shock Tier')
plt.xlabel('Workout Shock Tier')
plt.ylabel('Daily Calorie Allowance (%)')
show_fig()
plot_no += 1

---
## 5. Correlation Analysis

The heatmap below shows the statistical relationships between all numerical variables.
This helps identify:
- Which features are most strongly related to the target
- Potential multicollinearity issues
- Features that may be redundant

In [ ]:
# Correlation Heatmap
fig = plt.figure(figsize=(14, 10))
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title(f'{plot_no}. Correlation Heatmap of Nutritional, Consumer and Caloric Variables')
show_fig()
plot_no += 1

In [ ]:
# Correlation specifically with the regression target
target_corr = corr[TARGET_REGRESSION].sort_values(ascending=False)

print("Correlation with Net Caloric Surplus:")
print("-" * 40)
for feature, value in target_corr.items():
    if feature != TARGET_REGRESSION:
        print(f"  {feature:35s}: {value:+.4f}")

# Visualize top correlations
fig = plt.figure(figsize=(10, 6))
top_corr = target_corr.drop(TARGET_REGRESSION).sort_values(ascending=True)
top_corr.plot(kind='barh', color='steelblue')
plt.title(f'{plot_no}. Feature Correlation with Net Caloric Surplus')
plt.xlabel('Correlation Coefficient')
show_fig()
plot_no += 1

---
## 6. Feature Selection and Data Leakage Prevention

### Classification Task (Workout Shock Tier)
We use meal nutritional information and consumer biometrics as input features.

### Regression Task (Net Caloric Surplus)
We use only the raw nutritional and consumer features.

### Data Leakage Prevention
The following columns are **excluded** from input features because they are
derived from or directly related to the target variable:
- `thermic_effect_calories`
- `walking_burn_time_min`
- `cycling_burn_time_min`
- `running_burn_time_min`
- `hiit_burn_time_min`
- `daily_calorie_allowance_pct`

Using these would allow the model to "cheat" rather than learn meaningful patterns.

In [ ]:
# ============================================================
# CLASSIFICATION: Predict workout_shock_tier
# ============================================================

classification_features = [
    'fast_food_chain', 'meal_category', 'item_count',
    'total_calories', 'fat_grams', 'saturated_fat_grams',
    'carb_grams', 'sugar_grams', 'protein_grams', 'sodium_mg',
    'consumer_gender', 'consumer_age', 'consumer_weight_kg', 'fitness_level'
]

# Keep only columns that exist in the dataset
classification_features = [col for col in classification_features if col in df.columns]

X_clf = df[classification_features].copy()
y_clf = df[TARGET_CLASSIFICATION].copy()

print(f"Classification Task:")
print(f"  Features: {X_clf.shape[1]}")
print(f"  Samples:  {X_clf.shape[0]}")
print(f"  Classes:  {y_clf.nunique()}")
print(f"  Class distribution:\n{y_clf.value_counts().to_string()}")

In [ ]:
# ============================================================
# REGRESSION: Predict net_caloric_surplus
# ============================================================

regression_features = [
    'fat_grams', 'saturated_fat_grams', 'carb_grams',
    'sugar_grams', 'protein_grams', 'sodium_mg',
    'consumer_age', 'consumer_weight_kg'
]

X_reg = df[regression_features].copy()
y_reg = df[TARGET_REGRESSION].copy()

print(f"Regression Task:")
print(f"  Features: {X_reg.shape[1]}")
print(f"  Samples:  {X_reg.shape[0]}")
print(f"  Target range: {y_reg.min():.1f} to {y_reg.max():.1f}")
print(f"  Target mean:  {y_reg.mean():.1f}")

---
## 7. Train-Test Split

We split the data into training (80%) and testing (20%) sets.

- **Training set**: Used by the model to learn patterns
- **Testing set**: Used to evaluate performance on unseen data

For classification, we use stratified splitting to maintain class proportions.

In [ ]:
# Classification split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, stratify=y_clf, random_state=RANDOM_STATE
)

print("Classification Split:")
print(f"  Training: {X_train_clf.shape[0]} samples")
print(f"  Testing:  {X_test_clf.shape[0]} samples")

In [ ]:
# Regression split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)

print("Regression Split:")
print(f"  Training: {X_train_reg.shape[0]} samples")
print(f"  Testing:  {X_test_reg.shape[0]} samples")

---
## 8. Preprocessing Pipeline

We build a preprocessing pipeline that handles:
- **Numeric features**: Missing value imputation (median) + Standard Scaling
- **Categorical features**: Missing value imputation (mode) + One-Hot Encoding

This pipeline is embedded inside our model pipeline to prevent data leakage
and ensure consistent preprocessing.

In [ ]:
# Identify feature types for classification
num_features_clf = X_train_clf.select_dtypes(include=['number']).columns.tolist()
cat_features_clf = X_train_clf.select_dtypes(exclude=['number']).columns.tolist()

print("Numeric features:", num_features_clf)
print("Categorical features:", cat_features_clf)

def make_preprocessor_clf():
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    return ColumnTransformer(transformers=[
        ('num', numeric_transformer, num_features_clf),
        ('cat', categorical_transformer, cat_features_clf)
    ])

print("\nClassification preprocessor created.")

In [ ]:
# For regression, all features are numeric
num_features_reg = X_train_reg.select_dtypes(include=['number']).columns.tolist()

print("Numeric features for regression:", num_features_reg)

# Scale regression features
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

print("\nRegression features scaled.")
print(f"Scaled training shape: {X_train_reg_scaled.shape}")
print(f"Scaled testing shape:  {X_test_reg_scaled.shape}")

---
## 9. Classification Model: Predicting Workout Shock Tier

We train and compare two classification models:
1. **Random Forest Classifier** — Ensemble method, robust to overfitting
2. **Logistic Regression** — Linear baseline model

Models are evaluated using:
- Accuracy
- Precision, Recall, F1-Score
- Confusion Matrix
- 5-Fold Cross-Validation

In [ ]:
# Random Forest Pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', make_preprocessor_clf()),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    ))
])

# Logistic Regression Pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', make_preprocessor_clf()),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    ))
])

print("Classification pipelines created.")

In [ ]:
# Cross-validation comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_cv_scores = cross_val_score(rf_pipeline, X_train_clf, y_train_clf,
                                cv=cv, scoring='accuracy')
lr_cv_scores = cross_val_score(lr_pipeline, X_train_clf, y_train_clf,
                                cv=cv, scoring='accuracy')

cv_results = pd.DataFrame({
    'Model': ['Random Forest', 'Logistic Regression'],
    'Mean CV Accuracy': [rf_cv_scores.mean(), lr_cv_scores.mean()],
    'Std CV Accuracy': [rf_cv_scores.std(), lr_cv_scores.std()]
})

print("5-Fold Cross-Validation Results:")
print(cv_results.to_string(index=False))

# Visualize
fig = plt.figure(figsize=(8, 5))
sns.barplot(data=cv_results, x='Model', y='Mean CV Accuracy', palette='coolwarm')
plt.title(f'{plot_no}. Cross-Validation Accuracy Comparison')
plt.ylabel('Mean Accuracy')
plt.ylim(0, 1)
show_fig()
plot_no += 1

In [ ]:
# Train both models
rf_pipeline.fit(X_train_clf, y_train_clf)
lr_pipeline.fit(X_train_clf, y_train_clf)

# Predictions
rf_test_pred = rf_pipeline.predict(X_test_clf)
lr_test_pred = lr_pipeline.predict(X_test_clf)

# Test accuracy
rf_accuracy = accuracy_score(y_test_clf, rf_test_pred)
lr_accuracy = accuracy_score(y_test_clf, lr_test_pred)

print("Test Set Accuracy:")
print(f"  Random Forest:      {rf_accuracy:.4f}")
print(f"  Logistic Regression: {lr_accuracy:.4f}")

# Select best model
if rf_accuracy >= lr_accuracy:
    final_clf_model = rf_pipeline
    final_clf_name = "Random Forest"
else:
    final_clf_model = lr_pipeline
    final_clf_name = "Logistic Regression"

print(f"\nSelected Final Model: {final_clf_name}")

In [ ]:
# Detailed classification report
print(f"Classification Report — {final_clf_name}")
print("=" * 60)
print(classification_report(y_test_clf, final_clf_model.predict(X_test_clf)))

In [ ]:
# Confusion Matrix
final_clf_pred = final_clf_model.predict(X_test_clf)
cm = confusion_matrix(y_test_clf, final_clf_pred, labels=final_clf_model.classes_)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=final_clf_model.classes_
)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=20)
plt.title(f'{plot_no}. Confusion Matrix — {final_clf_name}')
show_fig()
plot_no += 1

In [ ]:
# Feature Importance from Random Forest
rf_classifier = rf_pipeline.named_steps['classifier']
rf_preprocessor = rf_pipeline.named_steps['preprocessor']

feature_names = rf_preprocessor.get_feature_names_out()
importances = pd.Series(rf_classifier.feature_importances_, index=feature_names)
top_importances = importances.sort_values(ascending=False).head(15)

fig = plt.figure(figsize=(10, 7))
top_importances.plot(kind='barh', color='steelblue')
plt.title(f'{plot_no}. Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.gca().invert_yaxis()
show_fig()
plot_no += 1

---
## 10. Regression Model: Predicting Net Caloric Surplus

We train two regression models:
1. **Gradient Boosting Regressor** (Machine Learning)
2. **Deep Neural Network** (Deep Learning using TensorFlow/Keras)

The target variable is `net_caloric_surplus` — the excess calories consumed
after accounting for the thermic effect of food.

In [ ]:
# Gradient Boosting Regressor
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=RANDOM_STATE
)

gb_model.fit(X_train_reg_scaled, y_train_reg)
y_pred_gb = gb_model.predict(X_test_reg_scaled)

# Evaluate
r2_gb = r2_score(y_test_reg, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test_reg, y_pred_gb))
mae_gb = mean_absolute_error(y_test_reg, y_pred_gb)

print("Gradient Boosting Regressor Results:")
print(f"  R² Score: {r2_gb:.4f}")
print(f"  RMSE:     {rmse_gb:.2f}")
print(f"  MAE:      {mae_gb:.2f}")

In [ ]:
# Actual vs Predicted for Gradient Boosting
fig = plt.figure(figsize=(10, 6))
plt.scatter(y_test_reg, y_pred_gb, alpha=0.7, color='teal')
lims = [
    min(float(y_test_reg.min()), float(y_pred_gb.min())),
    max(float(y_test_reg.max()), float(y_pred_gb.max()))
]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlabel('Actual Net Caloric Surplus')
plt.ylabel('Predicted Net Caloric Surplus')
plt.title(f'{plot_no}. Machine Learning: Actual vs Predicted Caloric Surplus')
plt.legend()
show_fig()
plot_no += 1

---
## 11. Deep Learning Model: Neural Network

We build a deep neural network with:
- 3 hidden layers (128, 64, 32 neurons)
- Dropout regularization to prevent overfitting
- EarlyStopping to halt training when validation loss stops improving
- ReduceLROnPlateau to reduce learning rate when training plateaus

In [ ]:
# Set random seed for reproducibility
tf.keras.utils.set_random_seed(RANDOM_STATE)

# Build the model
dl_model = Sequential([
    Input(shape=(X_train_reg_scaled.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.15),
    Dense(64, activation='relu'),
    Dropout(0.10),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression
])

# Compile
dl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-5
)

print("Neural Network Architecture:")
dl_model.summary()

In [ ]:
# Convert to float32 for TensorFlow
X_train_reg_scaled = X_train_reg_scaled.astype('float32')
X_test_reg_scaled = X_test_reg_scaled.astype('float32')
y_train_reg = y_train_reg.astype('float32')
y_test_reg = y_test_reg.astype('float32')

# Train the model
history = dl_model.fit(
    X_train_reg_scaled, y_train_reg,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\nTraining complete.")

In [ ]:
# Evaluate Neural Network
y_pred_dl = dl_model.predict(X_test_reg_scaled).flatten()

r2_dl = r2_score(y_test_reg, y_pred_dl)
rmse_dl = np.sqrt(mean_squared_error(y_test_reg, y_pred_dl))
mae_dl = mean_absolute_error(y_test_reg, y_pred_dl)

print("Deep Learning Model Results:")
print(f"  R² Score: {r2_dl:.4f}")
print(f"  RMSE:     {rmse_dl:.2f}")
print(f"  MAE:      {mae_dl:.2f}")

In [ ]:
# Actual vs Predicted for Deep Learning
fig = plt.figure(figsize=(10, 6))
plt.scatter(y_test_reg, y_pred_dl, alpha=0.7, color='purple')
lims = [
    min(float(y_test_reg.min()), float(y_pred_dl.min())),
    max(float(y_test_reg.max()), float(y_pred_dl.max()))
]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlabel('Actual Net Caloric Surplus')
plt.ylabel('Predicted Net Caloric Surplus')
plt.title(f'{plot_no}. Deep Learning: Actual vs Predicted Caloric Surplus')
plt.legend()
show_fig()
plot_no += 1

In [ ]:
# Plot training and validation loss
fig = plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title(f'{plot_no}. Neural Network Training and Validation Loss')
plt.legend()
show_fig()
plot_no += 1

---
## 12. Model Comparison Summary

We compare all models trained in this project to determine the best performer
for each task.

In [ ]:
# Final comparison table
comparison = pd.DataFrame({
    'Task': ['Classification', 'Classification', 'Regression', 'Regression'],
    'Model': ['Random Forest', 'Logistic Regression',
              'Gradient Boosting', 'Neural Network'],
    'Metric': ['Accuracy', 'Accuracy', 'R² Score', 'R² Score'],
    'Score': [rf_accuracy, lr_accuracy, r2_gb, r2_dl]
})

print("=" * 60)
print("FINAL MODEL COMPARISON")
print("=" * 60)
print(comparison.to_string(index=False))

# Visualize
fig = plt.figure(figsize=(10, 6))
sns.barplot(data=comparison, x='Model', y='Score', hue='Task', palette='viridis')
plt.title(f'{plot_no}. Final Model Comparison')
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.legend(title='Task Type')
show_fig()
plot_no += 1

---
## 13. Example Prediction

We demonstrate the trained model by predicting the workout shock tier
for a new, unseen fast-food order.

In [ ]:
# Example prediction for a new order
new_order = {
    'fast_food_chain': "McDonald's",
    'meal_category': 'Double Burger & Large Fry',
    'item_count': 3,
    'total_calories': 1100,
    'fat_grams': 56,
    'saturated_fat_grams': 20,
    'carb_grams': 110,
    'sugar_grams': 35,
    'protein_grams': 22,
    'sodium_mg': 1400,
    'consumer_gender': 'Male',
    'consumer_age': 30,
    'consumer_weight_kg': 75,
    'fitness_level': 'Sedentary'
}

new_order_df = pd.DataFrame([new_order])

# Classification prediction
predicted_tier = final_clf_model.predict(new_order_df)[0]
tier_probabilities = final_clf_model.predict_proba(new_order_df)[0]

print("=" * 60)
print("PREDICTION FOR NEW ORDER")
print("=" * 60)
print(f"\nOrder: {new_order['fast_food_chain']} — {new_order['meal_category']}")
print(f"Consumer: {new_order['consumer_age']}yo {new_order['consumer_gender']}, "
      f"{new_order['consumer_weight_kg']}kg, {new_order['fitness_level']}")
print(f"\nPredicted Workout Shock Tier: {predicted_tier}")
print(f"\nClass Probabilities:")
for category, prob in zip(final_clf_model.classes_, tier_probabilities):
    print(f"  {category:35s}: {prob:.2%}")

---
## 14. Conclusion and Key Findings

### Classification Task
The model successfully predicts the workout impact category of fast-food meals
based on nutritional content and consumer biometrics.

### Regression Task
The Gradient Boosting and Neural Network models accurately predict the net
caloric surplus, demonstrating that nutritional features are strong predictors
of caloric impact.

### Key Insights
- Total calories and fat grams are the strongest predictors of workout impact
- Fitness level significantly moderates the workout shock tier
- Higher protein content does not necessarily reduce caloric surplus
- Sedentary individuals face significantly higher workout shock tiers

### Why We Used Each Technique
| Technique | Purpose |
|---|---|
| Data Cleaning | Remove noise, duplicates, and inconsistencies |
| EDA | Understand patterns before modeling |
| Train-Test Split | Prevent overfitting, evaluate generalization |
| Scaling | Ensure features are on comparable ranges |
| One-Hot Encoding | Convert text categories to numerical format |
| Cross-Validation | Get robust performance estimates |
| Random Forest | Handle non-linear relationships |
| Neural Network | Capture complex patterns |
| EarlyStopping | Prevent overfitting in deep learning |

### Future Improvements
- Add hyperparameter tuning (GridSearchCV / Optuna)
- Add SHAP-based model explanations
- Deploy as a Streamlit web application
- Add more diverse fast-food chain data

In [ ]:
import pickle
from pathlib import Path

# Create models directory
Path('models').mkdir(exist_ok=True)

# Save classification pipeline
with open('models/fastfood_classification_model.pkl', 'wb') as f:
    pickle.dump(final_clf_model, f)

# Save regression model
with open('models/fastfood_regression_model.pkl', 'wb') as f:
    pickle.dump(gb_model, f)

print("Models saved successfully:")
print("  - models/fastfood_classification_model.pkl")
print("  - models/fastfood_regression_model.pkl")